# 原 Stage1 的 NVDA 指引实现状态输入预检

按 `source_preflight_plan.json` 只读既有Qlib Recorder，验证原Stage1 2011–2022训练日历和2023+完整账户日历的来源身份、09:29决策时点及已知状态。0拟合、0账户、0市场收益读取。

In [1]:
import torch
from pathlib import Path
import hashlib,json
import pandas as pd
import qlib
from qlib.workflow import R
from src.data.nvda_own_guidance_realization import align_realizations,FEATURE,FLAGS
root=Path.cwd();card=root/'research/stage1_nvda_guidance_receiver_20261022'
plan=json.loads((card/'source_preflight_plan.json').read_text())
objects={};summaries={}
for name,source in plan['sources'].items():
    path=root/source['card'];s=json.loads((path/'summary.json').read_text())
    assert s['recorder_id']==source['recorder_id']
    manifest=json.loads((path/'verification_manifest.json').read_text())
    hits=[x for x in manifest['entries'] if x['path'].endswith('/artifacts/'+source['artifact'])]
    assert len(hits)==1 and hits[0]['sha256']==source['sha256']
    assert hashlib.sha256((path/hits[0]['path']).read_bytes()).hexdigest()==source['sha256']
    summaries[name]=s
qlib.init(provider_uri=summaries['training_timing']['provider_uri'],region='us')
for name,source in plan['sources'].items():
    s=summaries[name]
    with R.uri_context(s['recorder_uri']):
        recorder=R.get_recorder(recorder_id=s['recorder_id'],experiment_id=s['experiment_id'])
        objects[name]=recorder.load_object(source['artifact'])
print('Three source SHA identities and Recorder reads passed.')

In [2]:
timing=objects['training_timing'];frame=objects['evaluation_calendar'];pairs=objects['guidance_pairs']
train_timing=timing.loc[timing.index<pd.Timestamp('2023-01-01'),'decision_at']
assert len(train_timing)==3020 and len(frame)==944
eval_index=pd.DatetimeIndex(frame.signal_date)
eval_trade=pd.to_datetime(frame.trade_date).dt.tz_localize('America/New_York')
eval_timing=pd.Series(eval_trade.array+pd.Timedelta(hours=9,minutes=29),index=eval_index)
assert eval_timing.dt.hour.eq(9).all() and eval_timing.dt.minute.eq(29).all()
assert (eval_timing.reset_index(drop=True)<pd.to_datetime(frame.decision_at).dt.tz_localize('America/New_York')).all()
train_features,train_trace,events=align_realizations(pairs,train_timing)
eval_features,eval_trace,_=align_realizations(pairs,eval_timing)
for features,trace in [(train_features,train_trace),(eval_features,eval_trace)]:
    assert features.loc[features[FLAGS[1]].eq(0),FEATURE].isna().all()
    assert features[FLAGS].isin([0.,1.]).all().all()
    valid=trace.source_available_at.notna()
    assert (pd.to_datetime(trace.loc[valid,'source_available_at'],utc=True)<=pd.to_datetime(trace.loc[valid,'decision_at'],utc=True)).all()
def coverage(x):
    return {'rows':len(x),'seen':int(x[FLAGS[0]].sum()),'comparable':int(x[FLAGS[1]].sum()),
            'known_preannounced':int(x[FLAGS[2]].sum()),'finite_numeric':int(x[FEATURE].notna().sum())}
result={'status':'conditional_input_coverage_verified_not_PIT','plan_sha256':hashlib.sha256((card/'source_preflight_plan.json').read_bytes()).hexdigest(),
        'source_recorder_ids':{key:value['recorder_id'] for key,value in summaries.items()},
        'training_calendar':coverage(train_features),'evaluation_calendar':coverage(eval_features),
        'train_first_decision':str(train_timing.min()),'train_last_decision':str(train_timing.max()),
        'evaluation_first_decision':str(eval_timing.min()),'evaluation_last_decision':str(eval_timing.max()),
        'source_realized_report_events':len(events),'historical_first_public_timestamp_verified':False,
        'all_updates_coverage_verified':False,'original_html_bytes_verified':False,
        'new_model_fits':0,'new_native_accounts':0,'market_outcome_reads':0,
        'interpretation':'Coverage and delayed alignment permit only a historical development design. Exact source/publication and future economic gates remain open.'}
(card/'source_preflight_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(result,ensure_ascii=False,indent=2))

In [3]:
from dataclasses import asdict
from src.model.stage1_nvda_guidance import Stage1NvdaGuidanceAllocator,initialization_contract
from src.workflow.pit_constituent_auxiliary_training_allocation import COMMON_FEATURES
implementation=json.loads((card/'implementation_preflight_plan.json').read_text())
assert hashlib.sha256((card/'source_preflight_plan.json').read_bytes()).hexdigest()==implementation['source_preflight_plan_sha256']
assert hashlib.sha256((root/implementation['model_adapter']).read_bytes()).hexdigest()==implementation['model_adapter_sha256']
source_card=root/implementation['training_source_card']
manifest=json.loads((source_card/'verification_manifest.json').read_text())
source=summaries['training_timing']
for name,digest in implementation['additional_training_artifacts_sha256'].items():
    entries=[e for e in manifest['entries'] if e['path'].endswith('/artifacts/'+name)]
    assert len(entries)==1 and entries[0]['sha256']==digest
    assert hashlib.sha256((source_card/entries[0]['path']).read_bytes()).hexdigest()==digest
with R.uri_context(source['recorder_uri']):
    recorder=R.get_recorder(recorder_id=source['recorder_id'],experiment_id=source['experiment_id'])
    base_features=recorder.load_object('inputs/features')
    original_models={seed:recorder.load_object('models/base_outer_'+str(seed)) for seed in implementation['expected']['seeds']}
full_guidance,full_trace,_=align_realizations(pairs,timing.decision_at)
pd.testing.assert_index_equal(base_features.index,full_guidance.index,check_names=False)
assert not set([*FLAGS,FEATURE]).intersection(base_features.columns)
joined=base_features.join(full_guidance)
train=joined.loc[joined.index<pd.Timestamp('2023-01-01')]
assert len(train)==implementation['expected']['pre2023_calendar_rows']
assert int(train[FLAGS[1]].sum())==implementation['expected']['guidance_comparable_calendar_rows']
assert len(COMMON_FEATURES)==implementation['expected']['auxiliary_common_features']
parity=[]
for seed,original in original_models.items():
    assert not original.extra_features and len(original.base_features)==implementation['expected']['base_primary_features']
    args=dict(base_features=original.base_features,spec={**asdict(original.spec),'root':str(root)},
              seed=original.seed,cutoff=str(original.cutoff),train_end=str(original.train_end.date()),
              minimum_observations=original.minimum_observations)
    b=Stage1NvdaGuidanceAllocator(**args,mode='neutral')
    c=Stage1NvdaGuidanceAllocator(**args,mode='observed')
    proof=initialization_contract(original,b,c)
    b._fit_extra_scaler(train);c._fit_extra_scaler(train)
    assert b.extra_scaler==c.extra_scaler
    left,right=b._extra_tensor(train),c._extra_tensor(train)
    assert torch.equal(left[:,:len(FLAGS)],right[:,:len(FLAGS)]) and left[:,-1].eq(0).all()
    assert int(right[:,-1].ne(0).sum())>0
    parity.append({'seed':seed,**proof,'calendar_scaler_rows':b.extra_scaler[FEATURE]['observations'],
                   'candidate_nonzero_numeric_rows':int(right[:,-1].ne(0).sum())})
result2={'status':'no_fit_adapter_and_source_mechanics_passed',
         'plan_sha256':hashlib.sha256((card/'implementation_preflight_plan.json').read_bytes()).hexdigest(),
         'source_recorder_id':source['recorder_id'],'seeds':parity,'base_primary_features':39,
         'common_features':len(COMMON_FEATURES),'train_calendar_rows':len(train),
         'train_calendar_comparable':int(train[FLAGS[1]].sum()),
         'model_fits':0,'native_accounts':0,'market_outcome_reads':0,
         'limitation':'Calendar-only scaler preview is not the formal mature-label training scaler. This preflight does not authorize fitting or account execution.'}
(card/'implementation_preflight_result.json').write_text(json.dumps(result2,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(result2,ensure_ascii=False,indent=2))

In [4]:
binding_plan=json.loads((card/'calendar_binding_plan.json').read_text())
bound={};source_summaries={}
for role,source in binding_plan['sources'].items():
    path=root/source['card'];summary=json.loads((path/'summary.json').read_text())
    assert summary['recorder_id']==source['recorder_id']
    manifest=json.loads((path/'verification_manifest.json').read_text())
    hits=[x for x in manifest['entries'] if x['path'].endswith('/artifacts/'+source['artifact'])]
    assert len(hits)==1 and hits[0]['sha256']==source['sha256']
    assert hashlib.sha256((path/hits[0]['path']).read_bytes()).hexdigest()==source['sha256']
    source_summaries[role]=summary
for role,source in binding_plan['sources'].items():
    summary=source_summaries[role]
    with R.uri_context(summary['recorder_uri']):
        recorder=R.get_recorder(recorder_id=summary['recorder_id'],experiment_id=summary['experiment_id'])
        bound[role]=recorder.load_object(source['artifact'])
model_predictions=bound['training_predictions'];funded=bound['Stage1_frame'][['signal_date','trade_date','exit_date']].reset_index(drop=True)
account=bound['Stage1_account']['replay'][['signal_date','trade_date','exit_date']].reset_index(drop=True)
pd.testing.assert_frame_equal(funded,account)
assert len(funded)==binding_plan['expected']['funded_intervals']==944
assert funded.signal_date.iloc[0].strftime('%Y-%m-%d')==binding_plan['expected']['first_signal']
assert funded.trade_date.iloc[0].strftime('%Y-%m-%d')==binding_plan['expected']['first_trade']
assert funded.signal_date.iloc[-1].strftime('%Y-%m-%d')==binding_plan['expected']['last_signal']
assert funded.trade_date.iloc[-1].strftime('%Y-%m-%d')==binding_plan['expected']['last_trade']
assert funded.exit_date.iloc[-1].strftime('%Y-%m-%d')==binding_plan['expected']['terminal_exit']
assert (funded.signal_date<funded.trade_date).all() and (funded.trade_date<funded.exit_date).all()
assert funded.exit_date.iloc[:-1].reset_index(drop=True).equals(funded.trade_date.iloc[1:].reset_index(drop=True))
assert pd.DatetimeIndex(eval_timing.index).equals(pd.DatetimeIndex(funded.signal_date))
assert pd.DatetimeIndex(eval_timing.dt.tz_localize(None).dt.normalize()).equals(pd.DatetimeIndex(funded.trade_date))
saved_funded_clock=bound['saved_training_timing'].loc[pd.DatetimeIndex(funded.signal_date),'decision_at']
pd.testing.assert_series_equal(saved_funded_clock,eval_timing,check_names=False)
assert str(saved_funded_clock.iloc[0])=='2023-01-04 09:29:00-05:00'
funded_signals=pd.DatetimeIndex(funded.signal_date)
for seed in binding_plan['expected'].get('seeds',[71,172,273]):
    predict=model_predictions[('base',seed)]
    assert len(predict)==binding_plan['expected']['all_model_prediction_rows']==946
    pd.testing.assert_index_equal(predict.index[:944],funded_signals,check_names=False)
    tail=[d.strftime('%Y-%m-%d') for d in predict.index[944:]]
    assert tail==binding_plan['expected']['unfunded_tail_predictions']
serial='\n'.join('|'.join(str(v.date()) for v in row) for row in funded[['signal_date','trade_date','exit_date']].itertuples(index=False,name=None))+'\n'
result3={'status':'exact_944_signal_trade_exit_binding_verified','plan_sha256':hashlib.sha256((card/'calendar_binding_plan.json').read_bytes()).hexdigest(),
         'funded_date_tuple_sha256':hashlib.sha256(serial.encode()).hexdigest(),
         'first_tuple':[str(x.date()) for x in funded.iloc[0]],'last_tuple':[str(x.date()) for x in funded.iloc[-1]],
         'prediction_rows_per_seed':946,'funded_prediction_rows':944,'unfunded_prediction_tail':tail,
         'native_account_calendar_matches':True,'conservative_0929_decision_matches_trade_date':True,
         'saved_training_timing_matches_all_944_trade_date_0929_clocks':True,
         'first_signal_saved_decision_at':str(saved_funded_clock.iloc[0]),
         'model_fits':0,'new_accounts':0,'return_values_used':0}
(card/'calendar_binding_result.json').write_text(json.dumps(result3,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(result3,ensure_ascii=False,indent=2))

## 中文结论

已从Qlib Recorder核对保守09:29来源时点：训练日历3020日中可比2865日，944个评价信号日均可比。三seed原Stage1与匹配B/C的初始输出一致，B只见来源标记、C另见标准化实现值。逐行日历绑定证明2023-01-03信号在2023-01-04决策/成交，原预测946行只有944行被资金账户使用。训练Dataset的独立无拟合预检又核对3014条成熟主样本、20741条辅助样本、2859条可比主样本；见同卡`training_dataset_preflight.json`。

训练及日历阶段：dataset_and_calendar_bound_design_only_no_fit。0优化器步、0新原生账户，尚无新增信息或净SOXX结果。独立设计复核要求的原价账本/控制器与B/C状态隔离仍须完成并复核；历史首次公开、更新完整性及原HTML字节仍未认证。

原价账户另以`baseline_parity_plan.json`完成一次Stage1 A 0bp参考复现：944行、1275笔订单、3778估值点与原生来源一致，NAV/现金/收益最大差0，且scikit-learn版本为源模型保存时的1.9.1。该步骤未执行B/C拟合或账户，当前卡片决定为native_account_source_and_A_parity_checked_no_fit。